# Mobile Money Agent Network — Data Audit and Cleanup

## 1. Load and inspect
We inspect dimensions, data types, missingness, date range, categories and sample records before cleaning.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from cleaning_pipeline import clean_agent_data

DATA = Path('.')
txns = pd.read_csv(DATA / 'agent_transactions.csv')
agents = pd.read_csv(DATA / 'agents.csv')
print('Transactions shape:', txns.shape)
print('Agents shape:', agents.shape)
display(txns.head())
display(txns.info())
display(txns.isna().sum().rename('missing_rows').to_frame())
print('Transaction types:', txns['txn_type'].value_counts(dropna=False).to_dict())
print('Raw date range:', txns['txn_date'].min(), 'to', txns['txn_date'].max())


## 2. Data-quality audit
The audit below quantifies the injected issues. Categories may overlap because one record can violate more than one dimension.

In [ ]:
audit = pd.DataFrame([
    {'issue':'Repeated txn_id values','dimension':'Uniqueness','column':'txn_id','rows_affected':int(txns.duplicated('txn_id').sum())},
    {'issue':'Missing amount values','dimension':'Completeness','column':'amount','rows_affected':int(txns['amount'].isna().sum())},
    {'issue':'N/A or non-numeric amount values','dimension':'Validity','column':'amount','rows_affected':int(pd.to_numeric(txns['amount'], errors='coerce').isna().sum())},
    {'issue':'Non-standard transaction labels','dimension':'Consistency','column':'txn_type','rows_affected':int((~txns['txn_type'].astype('string').str.strip().str.lower().isin(['deposit','withdrawal'])).sum())},
    {'issue':'Slash-formatted dates','dimension':'Validity / consistency','column':'txn_date','rows_affected':int(txns['txn_date'].astype('string').str.contains('/', na=False).sum())},
    {'issue':'KES 250,000 suspicious values','dimension':'Accuracy','column':'amount','rows_affected':int((pd.to_numeric(txns['amount'], errors='coerce') == 250000).sum())},
    {'issue':'Unknown agent IDs','dimension':'Referential integrity','column':'agent_id','rows_affected':int((~txns['agent_id'].isin(agents['agent_id'])).sum())},
])
display(audit)


## 3. Reusable cleaning pipeline
Duplicates are removed by `txn_id`; transaction labels are normalised; missing/invalid amounts are median-imputed and flagged; dates are parsed; IQR outliers are flagged but retained; a left join preserves unknown agents.

In [ ]:
clean_df, cleaning_log = clean_agent_data(txns, agents)
display(cleaning_log)
print('Raw transaction rows:', len(txns))
print('Rows after duplicate removal (expected):', txns['txn_id'].nunique())
print('Final cleaned rows:', len(clean_df))
print('Unknown-agent transactions:', int(clean_df['county'].isna().sum()))
print('Duplicate txn_id after cleaning:', int(clean_df['txn_id'].duplicated().sum()))
print('Transaction types:', sorted(clean_df['txn_type'].dropna().unique().tolist()))


## 4. Investigate outliers and imputation
The KES 250,000 observations are retained and flagged rather than automatically deleted: they may be genuine high-value transactions and require source-system/business verification. Median imputation is a reproducible placeholder for missing amounts; imputed rows remain explicitly identifiable and should be excluded from high-stakes value conclusions until verified.

In [ ]:
print('Imputed amounts:', int(clean_df['amount_imputed'].sum()))
print('IQR outliers:', int(clean_df['is_outlier'].sum()))
display(clean_df.loc[clean_df['is_outlier'], ['txn_id','agent_id','txn_type','amount','txn_date','is_outlier']].head(20))


## 5. County summary by transaction type
Only transactions with a known county contribute to county-level totals; unknown-agent transactions remain in the clean file for follow-up.

In [ ]:
known = clean_df.dropna(subset=['county']).copy()
county_summary = (known.groupby(['county','txn_type'])['amount']
                  .agg(total_value='sum', number_of_transactions='count', average_value='mean')
                  .reset_index())
display(county_summary)


## 6. Weekly transaction-value pivot

In [ ]:
known['txn_date'] = pd.to_datetime(known['txn_date'])
known['week'] = known['txn_date'].dt.to_period('W').astype(str)
weekly_pivot = pd.pivot_table(known, index='county', columns='week', values='amount', aggfunc='sum', fill_value=0)
display(weekly_pivot)


## 7. Export cleaned transactions

In [ ]:
Path('outputs').mkdir(exist_ok=True)
clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)
print('Saved outputs/agent_transactions_clean.csv')


## 8. Data Quality Report

### Accuracy
Amounts were converted to numeric values. IQR outliers are flagged, not deleted, because statistical extremeness alone does not prove an error. KES 250,000 transactions require source verification.

### Completeness
Missing and invalid amounts are median-imputed to preserve the transaction rows, with `amount_imputed` marking every affected record. Imputed amounts should not be treated as verified financial values.

### Consistency
Transaction labels are trimmed and case-normalised to exactly `Deposit` and `Withdrawal`. Slash-separated dates are normalised to ISO date format.

### Validity
Amounts are numeric, dates are parsed and exported as ISO `YYYY-MM-DD`, and the transaction-type domain is restricted to the two specified categories.

### Uniqueness
Duplicate transactions are identified by `txn_id`; the first occurrence is retained. The notebook reports the number removed and verifies no duplicate IDs remain.

### Referential integrity
Transactions are left-joined to the agent register. Unknown agent IDs are retained with a missing county so they can be investigated rather than silently discarded.

### Privacy and Kenya Data Protection Act 2019
If customer phone numbers were present, they would be personal data because they can identify or contact individuals. Access would be restricted by role, numbers masked or pseudonymised for analysis, data encrypted in transit and at rest, retention limited to the business purpose, and disclosures controlled in line with the Kenya Data Protection Act 2019 and applicable organisational policies.
